# stockout — exploration

The five questions in [`docs/questions.md`](../docs/questions.md) were committed before
this notebook existed. Answer them here, then write the conclusions in
[`docs/results.md`](../docs/results.md) with the number attached.

**Two rules.**

1. Do not add a sixth question after seeing the data. Anything interesting that turns
   up goes in results.md under *what we did not ask*, clearly marked as unplanned.
2. Report the hypotheses that lose. A results file where all five won is evidence the
   questions were written after the charts.

The package holds anything with one correct answer; the judgement calls belong here,
visible next to their output ([ADR 0006](../docs/decisions/0006-analysis-in-notebooks.md)).

---

**Every cell below runs, and none of them answers anything.** They run against the
committed synthetic sample, whose promotion calendar and weekday multipliers are
constants this repository chose; a finding drawn from it would be a finding about
`stockout.data.synth`. Point `DATA` at `data/raw/train.csv` once `python -m stockout
fetch` has run, re-run top to bottom, and the answers become real. Until then every
**Answer** heading says *unanswered*, and that is a data problem rather than an
oversight.

In [ ]:
from statistics import NormalDist

import matplotlib.pyplot as plt
import pandas as pd

from stockout.config import SAMPLE_PATH
from stockout.data.loaders import read_sales
from stockout.data.validate import calendar_gaps, null_profile, validate_sales
from stockout.evaluate.backtest import backtest, summarise
from stockout.evaluate.metrics import coverage_table, mase
from stockout.inventory.policy import critical_ratio
from stockout.inventory.simulate import frontier
from stockout.models import forecaster
from stockout.models.baselines import SeasonalNaive
from stockout.models.conformal import ConformalQuantileForecaster
from stockout.models.gbm import GbmForecaster, GbmQuantileForecaster
from stockout.plots import apply_style, save
from stockout.split.rolling import rolling_origin, split_frame

apply_style()

# Swap for data/raw/train.csv once `python -m stockout fetch` has run. Everything below
# works on the synthetic sample, but no finding may be reported from it.
DATA = SAMPLE_PATH
HORIZON = 42

sales = read_sales(DATA)
validate_sales(sales)
sales.head()

In [ ]:
trading = sales[sales['open'] == 1]
print(f'rows        {len(sales):,}')
print(f'stores      {sales.store.nunique():,}')
print(f'span        {sales.date.min().date()} to {sales.date.max().date()}')
print(f'trading     {len(trading) / len(sales):.1%} of rows')
display(null_profile(sales))
display(calendar_gaps(sales))

## Q1 — Does accuracy decay with horizon, and how fast?

**Hypothesis:** monotonic decay, steepest between 7 and 14 days.  
**Counts as a no:** WMAPE flat across horizons.

### Answer

*Unanswered — needs Rossmann.* The cell below draws the curve; on synthetic data it
describes the generator's own weekday cycle rather than retail.

In [ ]:
decay = pd.DataFrame(
    [
        {'horizon': h, **summarise(backtest(sales, SeasonalNaive, n_folds=3, horizon=h))}
        for h in (7, 14, 28, 42)
    ]
)

fig, ax = plt.subplots()
ax.plot(decay.horizon, decay.wmape, marker='o')
ax.set(
    xlabel='horizon (days)',
    ylabel='WMAPE',
    title='Q1 — accuracy against forecast distance',
)
save(fig, 'q1_horizon_decay')
decay

## Q2 — Does seasonal-naive beat a GBM on low-volume stores?

**Hypothesis:** the GBM wins overall but loses on the bottom volume quartile.  
**Counts as a no:** the GBM wins uniformly across quartiles.

> The synthetic sample draws every store's base level from one uniform range, so its
> quartiles are close together by construction, and four stores cannot populate four
> quartiles. This question needs Rossmann's 1115 stores to mean anything.

### Answer

*Unanswered — needs Rossmann.*

In [ ]:
# Two orderings matter here. Split first, then measure volume from `train` only: `trading`
# spans the whole file, so segmenting stores by their all-period mean would let the test
# window decide which stores count as low volume. That is leakage of exactly the kind this
# repo is built to refuse, and it is easy to miss because the leaked quantity is a label
# rather than a feature.
#
# And fit once, then score each quartile separately — refitting per quartile answers a
# different question (one model per segment, not one model serving segments of different
# size).
fold = rolling_origin(sales.date, n_folds=1, horizon=HORIZON)[-1]
train, test = split_frame(sales, fold)

volume = train[train['open'] == 1].groupby('store')['sales'].mean().rename('mean_sales')
quartile = pd.qcut(volume, 4, labels=['Q1 low', 'Q2', 'Q3', 'Q4 high'], duplicates='drop')

predictions = {
    name: forecaster(name, horizon=HORIZON)().fit(train).predict(test)
    for name in ('seasonal_naive', 'gbm')
}
scored = test.assign(quartile=test.store.map(quartile), **predictions)
scored = scored[scored['open'] == 1]


def score_segment(group):
    return pd.Series(
        {
            'stores': group.store.nunique(),
            'mean_sales': group.sales.mean(),
            'mase_gbm': mase(group.sales, group.gbm, y_baseline=group.seasonal_naive),
        }
    )


# Below 1 means the GBM beat the baseline inside that quartile. The hypothesis says the
# top row is the one that should sit above 1.
scored.groupby('quartile', observed=True).apply(score_segment, include_groups=False)

## Q3 — How much of the total error comes from a few days?

**Hypothesis:** heavily concentrated, clustered on holidays and promotion boundaries.  
**Counts as a no:** error spread evenly, so no special-case model is worth building.

### Answer

*Unanswered — needs Rossmann.* The generator has no event spikes beyond the ones it was
told to draw, so its concentration curve measures the draw.

In [ ]:
# Trading days only: a closed day is predicted zero and contributes exactly zero error, so
# including them would dilute the concentration by a seventh and flatter the curve.
errors = (scored.sales - scored.gbm).abs().sort_values(ascending=False)
share = (errors.cumsum() / errors.sum()).to_numpy()
days = [(i + 1) / len(share) for i in range(len(share))]

for cut in (0.05, 0.10, 0.20):
    carried = share[max(int(len(share) * cut) - 1, 0)]
    print(f'worst {cut:.0%} of days carry {carried:.1%} of the total absolute error')

fig, ax = plt.subplots()
ax.plot(days, share, label='observed')
ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1, label='error spread evenly')
ax.set(
    xlabel='share of test days, worst first',
    ylabel='cumulative share of absolute error',
    title='Q3 — where the error lives',
)
ax.legend()
save(fig, 'q3_error_concentration')

# The days themselves, so an answer can say *which* days rather than only how few.
scored.loc[errors.index[:10], ['date', 'store', 'sales', 'gbm', 'promo', 'school_holiday']]

## Q4 — Does the promotion lift persist afterwards, or reverse?

**Hypothesis:** a dip. Promotions pull demand forward rather than creating it.  
**Counts as a no:** post-promotion sales at or above a matched baseline.

Note this one is invisible to WMAPE and only shows up in the inventory simulation:
stocking to a promotion forecast overstocks the following week.

### Answer

*Unanswered — needs Rossmann.* The generator multiplies promotion days and models no
carry-over whatsoever, so a lift near zero here tests this cell rather than answering the
question. Anything far from zero on synthetic data means the matching is wrong.

In [ ]:
# A binary after-a-promotion flag does not survive contact with a promotion calendar that
# runs every other week: every non-promotion day then falls in the wake of one, no
# unaffected baseline is left, and the comparison divides by an empty set. So measure the
# *profile* instead — how sales move with the number of days since a promotion ended.
#
# The weekday cycle is divided out first, per store, so that a Saturday landing one day
# after a promotion cannot masquerade as a promotion effect. Every value below is relative
# to that store's mean non-promotion trading day of the same weekday, so 1.0 is neutral.
AFTER_DAYS = 7

window = sales.sort_values(['store', 'date']).copy()
is_last_promo_day = (window.promo == 1) & (window.groupby('store')['promo'].shift(-1) != 1)
window['last_promo_end'] = window.date.where(is_last_promo_day).groupby(window.store).ffill()
window['days_since_promo'] = (window.date - window.last_promo_end).dt.days

eligible = window[(window.promo == 0) & (window['open'] == 1)].copy()
weekday_mean = eligible.groupby(['store', 'day_of_week'])['sales'].transform('mean')
eligible['relative'] = eligible.sales / weekday_mean

wake = eligible[eligible.days_since_promo.between(1, AFTER_DAYS)]
profile = wake.groupby('days_since_promo')['relative'].agg(['mean', 'count'])

print(f'{int(is_last_promo_day.sum()):,} promotion windows, {len(wake):,} days in their wake')
print(f'first day after a promotion sells {profile.iloc[0, 0] - 1:+.2%} against its weekday')

fig, ax = plt.subplots()
ax.plot(profile.index, profile['mean'], marker='o')
ax.axhline(1.0, linestyle='--', linewidth=1)
ax.set(
    xlabel='days since the promotion ended',
    ylabel='sales relative to the same store and weekday',
    title='Q4 — the wake of a promotion',
)
save(fig, 'q4_promotion_wake')
profile

## Q5 — Does the newsvendor quantile beat stocking to the mean?

**Hypothesis:** the quantile policy reaches a higher fill rate at equal holding cost.  
**Counts as a no:** both policies land on the same frontier — the normal approximation
was good enough and the quantile models were not worth building.

This is the question the repository is named after, and a **no** is the most interesting
outcome available.

> Three policies are priced, not two. `mean_plus_z` is the textbook safety stock
> [ADR 0007](../docs/decisions/0007-quantiles-not-point-forecast-plus-z-score.md) argues
> against, `quantile` is the raw pinball fit, and `conformal` is that fit with the
> calibration of [ADR 0009](../docs/decisions/0009-conformal-calibration-not-a-recalibrated-loss.md)
> applied. The third column exists because the second one under-covers, and a comparison
> between two policies where one is miscalibrated is a comparison of calibration. Read the
> coverage cell below before believing any curve.

### Answer

*Unanswered — needs Rossmann.*

In [ ]:
# The target service level, derived from the cost pair rather than tuned.
target = critical_ratio()
level = f'{target}'

# Sigma for the normal policy is measured on a held-out tail rather than in sample, so the
# textbook rule is not handicapped by a residual spread it has already been shown. The
# conformal model gives up the same days for the same reason, which is what keeps this a
# fair comparison rather than a rigged one.
cutoff = train.date.max() - pd.Timedelta(days=HORIZON)
inner, tail = train[train.date <= cutoff], train[train.date > cutoff]
tail_open = tail['open'] == 1
probe = GbmForecaster(horizon=HORIZON).fit(inner)
sigma = float((tail.sales[tail_open] - probe.predict(tail)[tail_open]).std())

point_model = GbmForecaster(horizon=HORIZON).fit(train)
quantile_model = GbmQuantileForecaster(horizon=HORIZON).fit(train)
conformal_model = ConformalQuantileForecaster(horizon=HORIZON).fit(train)

# Inventory is held per store, so simulate one. Averaging a fill rate across a quiet
# shop and a busy one describes neither of them.
store = int(test.store.iloc[0])
rows = test.store == store

policies = pd.DataFrame(
    {
        'mean_plus_z': (
            point_model.predict(test) + NormalDist().inv_cdf(target) * sigma
        ).clip(lower=0.0),
        'quantile': quantile_model.predict_quantiles(test)[level],
        'conformal': conformal_model.predict_quantiles(test)[level],
    }
)

curve = frontier(test.sales[rows], policies[rows])
curve.insert(0, 'policy', list(policies.columns))
print(f'target quantile {target:.2f} · sigma {sigma:,.0f} · store {store}')
curve.drop(columns='quantile')

In [ ]:
# Calibration, before either frontier is believed. A well-calibrated 0.9 is exceeded 10%
# of the time; one that covers 0.99 is not conservative, it is wrong, and it carries stock
# nobody needed. Trading days only — a shut store is predicted zero and covered trivially.
open_rows = test['open'] == 1
actual = test.sales[open_rows]

calibration = pd.concat(
    [
        coverage_table(actual, model.predict_quantiles(test)[open_rows]).assign(model=name)
        for name, model in (
            ('gbm_quantile', quantile_model),
            ('gbm_conformal', conformal_model),
        )
    ],
    ignore_index=True,
)

print(f'{conformal_model.calibration_rows:,} rows in the calibration window')
print(f'saturated levels: {conformal_model.saturated_quantiles or None}')
calibration.pivot(index='quantile', columns='model', values='gap')

In [ ]:
# The decision layer under a delivery lag (ADR 0010, in docs/decisions/). Not one of the
# five questions, and it belongs beside Q5 because it changes that question's shape: once
# stock takes a week to arrive, holding is charged on every day of the protection interval
# and shortage only once, so the cost-minimising level stops being the critical ratio and
# starts being the cheapest one on the grid.
quantiles = conformal_model.predict_quantiles(test)

pipeline = pd.concat(
    [
        frontier(test.sales[rows], quantiles[rows], lead_time_days=lead).assign(lead_time=lead)
        for lead in (0, 7)
    ],
    ignore_index=True,
)
pipeline[['lead_time', 'quantile', 'fill_rate', 'mean_on_hand', 'mean_on_order', 'total_cost']]

---

## Before closing this notebook

- [ ] Every answer above written, with a number
- [ ] Conclusions copied into [`docs/results.md`](../docs/results.md)
- [ ] *What didn't work* filled in — a dead end costs a day either way, and writing it
      down is the only way that day buys anything
- [ ] Charts saved via `save(fig, "qN_name")` into `reports/` (gitignored, regenerated)
- [ ] Anything reused twice promoted into the package, where it picks up a test

Three things have already made that last trip: `metrics.coverage_table` came out of the
calibration cell, `models.conformal` came out of what the calibration cell showed, and
`simulate`'s delivery pipeline came out of the cell after it.